# Single-threshold directional change and its filtered event-time HMM

This notebook rebuilds the one-threshold event ledger from `1_threshold_dc_explore.ipynb`, then fits a two-state, filtered, event-time HMM on $\log R$ and audits what its states mean (Audits 1–7, plus the model-free Audit 6b).

For each completed extreme-to-extreme trend:

$$\mathrm{TMV} = \frac{|P_{\mathrm{EXT},n} - P_{\mathrm{EXT},n-1}|}{P_{\mathrm{EXT},n-1}\,\theta}$$

$$T = t_{\mathrm{EXT},n} - t_{\mathrm{EXT},n-1}$$

$$R = \frac{\mathrm{TMV}\,\theta}{T}$$

The ending extreme is identified retrospectively. Therefore, TMV, T, and R are not available at `ext_end_idx`; they first become knowable at the later DC confirmation, `dcc_idx`. The notebook records that timestamp as `information_time`.

In [1]:
# Reproducibility: pin thread counts before NumPy or PyTorch is imported, so EM sums run in a fixed order.
import os

HMM_THREADS = 1
for _variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_variable] = str(HMM_THREADS)

from enum import StrEnum

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from dc_basics import prepare_event_ledger
from dc_basics.core import log_r, parse_dc_events
from dc_basics.data import resolve_data_root
from dc_basics.hmm import DistType, InferenceMode, create_simple_hmm_instance
from dc_basics.hmm_support import SEMANTIC_R_STATE_NAMES, configure_reproducible_threads, semantic_r_state_order
from dc_basics.plotting import (
    plot_dc_indicator_distributions,
    plot_directional_change,
    plot_hmm_event_time_strip,
    plot_hmm_log_r_distributions,
    plot_hmm_next_event_outcomes,
    plot_hmm_next_state_calibration,
    plot_hmm_probability_vs_log_r,
    plot_hmm_recent_event_casebook,
    plot_hmm_states_on_price,
    plot_hmm_tmv_t_space,
)

print(configure_reproducible_threads(HMM_THREADS))

{'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1', 'torch_num_threads': 1}


## Configuration

In [2]:
DATA_ROOT = resolve_data_root()  # $DC_DATA_ROOT or the git-ignored .dc_data_root file
SYMBOL = "EURUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"

THETA = 0.00112
N_PRICES = 5_000

print(f"Symbol: {SYMBOL}")
print(f"Data: {DATA_PATH}")
print(f"Threshold: {THETA:.4%}")
print(f"Chart window: {N_PRICES:,} prices")

Symbol: EURUSD.r
Data: D:\data_dump\market_data\raw\FXPIG-Server\5\EURUSD.r.parquet
Threshold: 0.1120%
Chart window: 5,000 prices


## Load and audit the bars

Directional-change parsing is path-dependent, so the price series must be chronological, unique, finite, and strictly positive. The cell below refuses unsorted, duplicated, non-finite, or non-positive prices rather than silently repairing them. If `time` is stored as a column rather than the Parquet index, it is promoted to the index explicitly.

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))

raw = raw.sort_index()
if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no 'close' column.")

prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all():
    raise ValueError("Close contains NaN or infinite values.")
if prices.le(0.0).any():
    raise ValueError("DC thresholds require strictly positive prices.")

print(f"Bars: {len(prices):,}")
print(f"Start: {prices.index[0]}")
print(f"End: {prices.index[-1]}")
print(f"Timezone: {prices.index.tz}")
raw.tail()

Bars: 258,848
Start: 2023-05-22 14:00:00+02:00
End: 2026-11-09 13:30:00+02:00
Timezone: pytz.FixedOffset(120)


,open,high,low,close,tick_volume,spread
date,,,,,,
2026-11-09 13:10:00+02:00,1.11924,1.11959,1.11922,1.11949,191.0,6
2026-11-09 13:15:00+02:00,1.11948,1.11984,1.11939,1.11958,268.0,6
2026-11-09 13:20:00+02:00,1.11956,1.11956,1.11899,1.11905,194.0,6
2026-11-09 13:25:00+02:00,1.11906,1.11930,1.11900,1.11911,109.0,6
2026-11-09 13:30:00+02:00,1.11912,1.11933,1.11908,1.11921,134.0,6


## Calculate completed directional-change events

`parse_dc_events` returns one row per completed trend. `prepare_event_ledger` then attaches the event identity and `information_time = dcc_idx`, because that confirmation bar is when the ending extreme and its completed TMV, T, and R become knowable. It also checks the ledger's causal invariants: no two events share a confirmation bar, no ending extreme lies after its information time, and $R = \mathrm{TMV}\,\theta/T$ on every row.

The ledger states direction explicitly. `direction` (identical to `completed_leg_direction`) is the leg that **ended**. `dcc_direction` is the direction of the confirmation itself, and `new_live_mode` is the mode that is live from `information_time` on. A completed `up` leg is confirmed by a downward DCC and switches the live mode down.

The library's string ID is kept as `ledger_event_id` (for example `E17`). This lab also keeps the plain event number as `event_id`, because its plots place events on a numeric event-order axis.

In [4]:
source_events = parse_dc_events(prices=prices, theta=THETA)
if source_events.empty:
    raise ValueError("No completed DC events were found. Check the data and threshold.")

events = prepare_event_ledger(source_events, prefix="E", scale="single", theta=THETA)
events = events.rename(columns={"event_id": "ledger_event_id"})
events.insert(0, "event_id", np.arange(1, len(events) + 1, dtype=np.int64))

event_columns = [
    "event_id", "threshold", "direction", "dcc_direction", "new_live_mode",
    "ext_start_idx", "ext_start_price", "ext_end_idx", "ext_end_price", "dcc_idx", "dcc_price",
    "information_time", "tmv", "t", "r",
]

print(f"Completed events: {len(events):,}")
events[event_columns].tail(10)

Completed events: 6,372

,event_id,threshold,direction,dcc_direction,new_live_mode,ext_start_idx,ext_start_price,ext_end_idx,ext_end_price,dcc_idx,dcc_price,information_time,tmv,t,r
6362,6363,0.00112,up,down,down,2026-11-05 16:35:00+02:00,1.12038,2026-11-05 17:45:00+02:00,1.12315,2026-11-05 18:05:00+02:00,1.12148,2026-11-05 18:05:00+02:00,2.207478,14,0.000177
6363,6364,0.00112,down,up,up,2026-11-05 17:45:00+02:00,1.12315,2026-11-05 18:05:00+02:00,1.12148,2026-11-05 18:45:00+02:00,1.12318,2026-11-05 18:45:00+02:00,1.327580,4,0.000372
6364,6365,0.00112,up,down,down,2026-11-05 18:05:00+02:00,1.12148,2026-11-05 19:50:00+02:00,1.12399,2026-11-05 20:40:00+02:00,1.12257,2026-11-05 20:40:00+02:00,1.998316,21,0.000107
6365,6366,0.00112,down,up,up,2026-11-05 19:50:00+02:00,1.12399,2026-11-05 23:05:00+02:00,1.12153,2026-11-06 01:25:00+02:00,1.12299,2026-11-06 01:25:00+02:00,1.954135,39,0.000056
6366,6367,0.00112,up,down,down,2026-11-05 23:05:00+02:00,1.12153,2026-11-06 02:00:00+02:00,1.12330,2026-11-06 02:55:00+02:00,1.12185,2026-11-06 02:55:00+02:00,1.409108,35,0.000045
6367,6368,0.00112,down,up,up,2026-11-06 02:00:00+02:00,1.12330,2026-11-06 09:30:00+02:00,1.11698,2026-11-06 10:00:00+02:00,1.11844,2026-11-06 10:00:00+02:00,5.023464,90,0.000063
6368,6369,0.00112,up,down,down,2026-11-06 09:30:00+02:00,1.11698,2026-11-06 10:30:00+02:00,1.11892,2026-11-06 11:10:00+02:00,1.11756,2026-11-06 11:10:00+02:00,1.550738,12,0.000145
6369,6370,0.00112,down,up,up,2026-11-06 10:30:00+02:00,1.11892,2026-11-06 16:20:00+02:00,1.11324,2026-11-06 16:35:00+02:00,1.11450,2026-11-06 16:35:00+02:00,4.532432,70,0.000073
6370,6371,0.00112,up,down,down,2026-11-06 16:20:00+02:00,1.11324,2026-11-06 17:20:00+02:00,1.11589,2026-11-06 17:55:00+02:00,1.11450,2026-11-06 17:55:00+02:00,2.125392,12,0.000198
6371,6372,0.00112,down,up,up,2026-11-06 17:20:00+02:00,1.11589,2026-11-06 23:55:00+02:00,1.11367,2026-11-09 02:10:00+02:00,1.11505,2026-11-09 02:10:00+02:00,1.776289,79,0.000025


## Visualize the last N prices

The close is shown with completed extreme-to-extreme trends and their later DC confirmations. A turning-point marker describes where an extreme occurred; its hover text shows when that extreme became knowable. The final unfinished event is intentionally absent because `parse_dc_events` emits completed events only.

In [5]:
fig = plot_directional_change(prices=prices, events=events, n_prices=N_PRICES, title=f"{SYMBOL} | theta={THETA:.4%}",)
fig.show()

## Descriptive EDA of TMV, T, and R

The mean answers the average question, but these event indicators are typically right-skewed. Median and tail quantiles are reported beside it so a small number of extreme events cannot silently determine the interpretation. These are descriptive event-level results: consecutive DC events are path-dependent, so the rows must not be treated as independent observations for significance testing.

In [6]:
INDICATORS = ["tmv", "t", "r"]
QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

overall_summary = events[INDICATORS].describe(percentiles=QUANTILES).T
overall_summary.index.name = "indicator"
overall_summary.style.format(precision=6)

,count,mean,std,min,10%,25%,50%,75%,90%,95%,99%,max
indicator,,,,,,,,,,,,
tmv,6372.000000,2.354100,1.418816,1.000001,1.142658,1.373744,1.901321,2.858435,4.136289,5.114125,7.615418,14.163872
t,6372.000000,40.595417,49.453013,1.000000,5.000000,10.000000,22.000000,51.000000,101.000000,146.450000,231.580000,547.000000
r,6372.000000,0.000187,0.000276,0.000005,0.000027,0.000050,0.000102,0.000213,0.000408,0.000622,0.001384,0.005896


### Direction-conditioned averages

The event count must be read with every directional comparison. A difference between up and down averages is descriptive here; it is not yet evidence of a stable or tradable asymmetry.

In [7]:
direction_summary = events.groupby("direction", sort=False).agg(
    event_count=("event_id", "size"),
    tmv_mean=("tmv", "mean"),
    tmv_median=("tmv", "median"),
    t_mean=("t", "mean"),
    t_median=("t", "median"),
    r_mean=("r", "mean"),
    r_median=("r", "median"),
)
direction_summary.style.format({
    "event_count": "{:,}",
    "tmv_mean": "{:.6f}",
    "tmv_median": "{:.6f}",
    "t_mean": "{:.2f}",
    "t_median": "{:.2f}",
    "r_mean": "{:.6e}",
    "r_median": "{:.6e}",
})

,event_count,tmv_mean,tmv_median,t_mean,t_median,r_mean,r_median
direction,,,,,,,
up,"3,186",2.362439,1.901551,40.86,22.00,1.877791e-04,1.027453e-04
down,"3,186",2.345762,1.900588,40.33,23.00,1.858110e-04,1.003787e-04


### Direction-conditioned distributions

Each histogram is normalized within direction, so up and down shapes remain comparable even if their event counts differ. The initial x-axis view ends at the full-sample 99th percentile; Plotly's **Autoscale** button reveals all tail observations.

In [8]:
distribution_fig = plot_dc_indicator_distributions(events)
distribution_fig.show()

# Single-threshold HMM

The HMM receives one observation per completed DC event: $\log(R)$. This remains a one-feature R model; the logarithm compresses R's strong right tail for the Gaussian emission baseline.

This is an **event-time** model. Its posterior updates only when a completed event becomes available at `information_time = dcc_idx`. Forward-filling a state across five-minute bars below is a visualization of the latest available belief, not bar-by-bar HMM inference.

The model is fitted on the first 80% of events. Filtering is causal in its observations, but filtered values inside the training interval are still in-sample because the fitted parameters use that complete interval. Only the final 20% is a genuinely held-out causal sequence.

## Prepare the single R observation and chronological split

In [9]:
HMM_DISTRIBUTION = DistType.NORMAL
N_HMM_STATES = 2
TRAIN_FRACTION = 0.80
HMM_RANDOM_STATE = 100
HMM_MAX_ITER = 500
HMM_N_RESTARTS = 5

hmm_events = events.copy()
hmm_events["log_r"] = log_r(hmm_events)

train_event_count = int(len(hmm_events) * TRAIN_FRACTION)
if train_event_count < 20 or len(hmm_events) - train_event_count < 2:
    raise ValueError("The chronological train/OOS split contains too few events.")

x_all = hmm_events[["log_r"]].to_numpy(dtype=np.float64)
x_train = x_all[:train_event_count]
oos_start_time = hmm_events.iloc[train_event_count]["information_time"]

print(f"Observation: log(R), shape={x_all.shape}")
print(f"Training events: {train_event_count:,}")
print(f"OOS events: {len(hmm_events) - train_event_count:,}")
print(f"First OOS information time: {oos_start_time}")

Observation: log(R), shape=(6372, 1)
Training events: 5,097
OOS events: 1,275
First OOS information time: 2026-01-06 05:55:00+02:00


## Fit and filter

State indices emitted by an HMM are arbitrary. After fitting, the states are named with one shared rule, `semantic_r_state_order` from `dc_basics.hmm_support`: the state with the lower filtered-probability-weighted mean of **raw R** over the **training** events is `slow_R`; the other is `fast_R`. The same rule names the states in `1_threshold_dc_explore_hmm_dist.ipynb` and in Chapter 3. It uses training rows only, never OOS outcomes.

In [10]:
hmm_model = create_simple_hmm_instance(
    dist_type=HMM_DISTRIBUTION,
    n_states=N_HMM_STATES,
    random_state=HMM_RANDOM_STATE,
    max_iter=HMM_MAX_ITER,
    inference_mode=InferenceMode.FILTERING,
    covariance_type="diag",
    min_cov=1e-6,
    n_restarts=HMM_N_RESTARTS,
)
hmm_model.fit(x_train)

filtered_probabilities = hmm_model.predict_proba(x_all)
if filtered_probabilities.shape != (len(hmm_events), N_HMM_STATES):
    raise ValueError(f"Unexpected probability shape: {filtered_probabilities.shape}")
np.testing.assert_allclose(filtered_probabilities.sum(axis=1), 1.0, rtol=1e-10, atol=1e-10)

emission_locations = np.array([
    float(np.ravel(parameters["means"])[0]) for parameters in hmm_model.parameters
])
raw_state_order, training_weighted_mean_r = semantic_r_state_order(
    filtered_probabilities[:train_event_count], hmm_events["r"].to_numpy()[:train_event_count]
)
state_names = np.array(SEMANTIC_R_STATE_NAMES)
raw_to_state_name = {int(raw_state): state_names[rank] for rank, raw_state in enumerate(raw_state_order)}
semantic_probabilities = filtered_probabilities[:, raw_state_order]
semantic_state_positions = np.argmax(semantic_probabilities, axis=1)

hmm_events["raw_state"] = np.argmax(filtered_probabilities, axis=1)
hmm_events["state_name"] = state_names[semantic_state_positions]
hmm_events["state_probability"] = semantic_probabilities.max(axis=1)
hmm_events["p_slow_r"] = semantic_probabilities[:, 0]
hmm_events["p_fast_r"] = semantic_probabilities[:, 1]
hmm_events["sample"] = "train"
hmm_events.iloc[train_event_count:, hmm_events.columns.get_loc("sample")] = "oos"

print(f"Inference mode: {hmm_model.inference_mode}")
print(f"Raw-state mapping: {raw_to_state_name}")

Inference mode: filtering
Raw-state mapping: {1: np.str_('slow_R'), 0: np.str_('fast_R')}


## Inspect what the states learned

The emission location is a mean in log-R space. The semantic order comes from the training-weighted mean of raw R, so `slow_R` need not have the lower log-R emission mean when the state variances differ a lot; the table shows both. Its exponential is a geometric characteristic R, not the arithmetic mean R. The transition matrix advances one completed DC event per step.

In [11]:
emission_summary = pd.DataFrame({
    "state_name": state_names,
    "raw_state": raw_state_order,
    "emission_mean_log_r": emission_locations[raw_state_order],
    "emission_geometric_r": np.exp(emission_locations[raw_state_order]),
    "training_weighted_mean_r": training_weighted_mean_r,
})

semantic_transition_matrix = pd.DataFrame(
    hmm_model.transition_prob()[np.ix_(raw_state_order, raw_state_order)],
    index=pd.Index(state_names, name="from_state"),
    columns=pd.Index(state_names, name="to_state"),
)

state_event_summary = hmm_events.groupby(["sample", "state_name"], sort=False).agg(
    event_count=("event_id", "size"),
    mean_r=("r", "mean"),
    median_r=("r", "median"),
    mean_t=("t", "mean"),
    median_t=("t", "median"),
    mean_tmv=("tmv", "mean"),
    mean_probability=("state_probability", "mean"),
)

display(emission_summary.style.format(precision=6))
display(semantic_transition_matrix.style.format(precision=4))
display(state_event_summary.style.format(precision=6))

,state_name,raw_state,emission_mean_log_r,emission_geometric_r,training_weighted_mean_r
0,slow_R,1,-9.557833,0.000071,0.000097
1,fast_R,0,-8.044970,0.000321,0.000425


to_state,slow_R,fast_R
from_state,,
slow_R,0.9217,0.0783
fast_R,0.1801,0.8199


## Map filtered event states onto price

Each update is indexed by its DCC information timestamp and forward-filled until the next completed event. The state therefore means **the latest event-time belief currently available**. It does not update between DCCs, and no state is assigned before the first completed event.

In [12]:
event_state_updates = hmm_events[[
    "information_time", "event_id", "state_name", "state_probability",
    "p_slow_r", "p_fast_r",
]].copy()
if event_state_updates["information_time"].duplicated().any():
    raise ValueError("Multiple HMM updates share an information timestamp.")

event_state_updates = event_state_updates.rename(columns={
    "information_time": "state_information_time",
    "event_id": "source_event_id",
}).set_index("state_information_time", drop=False)

bar_hmm_states = event_state_updates.reindex(prices.index).ffill()
first_information_time = event_state_updates.index[0]
assert bar_hmm_states.loc[bar_hmm_states.index < first_information_time, "state_name"].isna().all()

hmm_price_fig = plot_hmm_states_on_price(prices=prices, bar_states=bar_hmm_states, hmm_events=hmm_events,
                                         n_prices=20000, title=f"{SYMBOL} | theta={THETA:.4%} | filtered HMM on log(R)",
                                         oos_start_time=oos_start_time)
hmm_price_fig.show()

## Event-weighted versus bar-weighted occupancy

The same filtered states can be counted in two ways.

- **Event-weighted** occupancy counts completed events: each DCC update counts once.
- **Bar-weighted** occupancy counts the five-minute bars that display the state after forward-filling. A state that tends to be followed by a long wait for the next DCC owns more bars than events.

Neither count is more "true"; they answer different questions, and the chart above shows only the bar-weighted view. Bars before the first completed event carry no state and are excluded. The last state is carried to the end of the data, so the final interval is right-censored. Bars are assigned to train or OOS by bar time relative to the first OOS information time.

In [13]:
bar_state_rows = bar_hmm_states.loc[bar_hmm_states["state_name"].notna(), ["state_name"]].copy()
bar_state_rows["sample"] = np.where(bar_state_rows.index >= oos_start_time, "oos", "train")

occupancy_index = pd.MultiIndex.from_product([["train", "oos"], list(state_names)], names=["sample", "state_name"])
state_occupancy = pd.DataFrame({
    "event_count": hmm_events.groupby(["sample", "state_name"]).size().reindex(occupancy_index, fill_value=0),
    "bar_count": bar_state_rows.groupby(["sample", "state_name"]).size().reindex(occupancy_index, fill_value=0),
})
for count_column, share_column in (("event_count", "event_share"), ("bar_count", "bar_share")):
    state_occupancy[share_column] = (
        state_occupancy[count_column] / state_occupancy.groupby(level="sample")[count_column].transform("sum")
    )
state_occupancy["bars_per_event"] = state_occupancy["bar_count"] / state_occupancy["event_count"].replace(0, np.nan)

assert state_occupancy["event_count"].sum() == len(hmm_events)
assert state_occupancy["bar_count"].sum() == int(bar_hmm_states["state_name"].notna().sum())
state_occupancy.style.format({
    "event_count": "{:,}",
    "bar_count": "{:,}",
    "event_share": "{:.2%}",
    "bar_share": "{:.2%}",
    "bars_per_event": "{:.1f}",
})

# HMM coherence audits

These audits test the specific contention raised by the price chart: do visually similar DC events receive different states for a defensible sequence-dependent reason, or is the model behaving like an incoherent rare-tail classifier?

No model is refitted here. All results use the same NORMAL HMM, log-R observation, chronological split, filtered probabilities, and state mapping created above.

In [14]:
audit_events = hmm_events.copy()
audit_events["previous_p_fast_r"] = audit_events["p_fast_r"].shift(1)
audit_events[[
    "event_id", "information_time", "sample", "r", "log_r",
    "previous_p_fast_r", "p_fast_r", "state_name",
]].tail()

,event_id,information_time,sample,r,log_r,previous_p_fast_r,p_fast_r,state_name
6367,6368,2026-11-06 10:00:00+02:00,oos,0.000063,-9.680117,0.004385,0.008494,slow_R
6368,6369,2026-11-06 11:10:00+02:00,oos,0.000145,-8.840603,0.008494,0.077374,slow_R
6369,6370,2026-11-06 16:35:00+02:00,oos,0.000073,-9.531663,0.077374,0.022328,slow_R
6370,6371,2026-11-06 17:55:00+02:00,oos,0.000198,-8.525377,0.022328,0.173498,slow_R
6371,6372,2026-11-09 02:10:00+02:00,oos,0.000025,-10.589348,0.173498,0.001786,slow_R


## Audit 1: log-R distributions by state

If the state names are meaningful, `fast_R` should occupy materially higher log-R values than `slow_R`. Overlap is expected because an HMM also uses the preceding belief and transition probabilities. Compare training and OOS rather than accepting in-sample separation alone.

In [15]:
log_r_audit_fig = plot_hmm_log_r_distributions(audit_events)
log_r_audit_fig.show()

## Audit 2: TMV–T geometry

Because $R = \mathrm{TMV}\,\theta/T$, equal R lies along a diagonal relationship in TMV–T space. Both axes are logarithmic. This chart reveals whether `fast_R` genuinely corresponds to high movement per unit time, even when two price-chart segments look similar.

In [16]:
tmv_t_audit_fig = plot_hmm_tmv_t_space(audit_events)
tmv_t_audit_fig.show()

## Audit 3: observation versus filtered posterior

The x-axis is the current event's log-R observation; the y-axis is the resulting filtered probability of `fast_R`. Hover `previous p(fast_R)` to see the sequence context entering the update. Vertical spread at nearly equal log-R is the direct footprint of the transition prior and previous belief.

In [17]:
probability_audit_fig = plot_hmm_probability_vs_log_r(audit_events)
probability_audit_fig.show()

## Audit 4: event-time state sequence

This removes bar-duration weighting entirely. Every completed DC event occupies one horizontal step, regardless of whether its state was later carried across five bars or five hundred bars on the price chart.

In [18]:
event_time_audit_fig = plot_hmm_event_time_strip(audit_events, train_event_count=train_event_count)
event_time_audit_fig.show()

## Audit 5: nearest OOS observations assigned to different states

Each row pairs an OOS `fast_R` event with the `slow_R` event having the nearest log-R. If current observations are almost identical but previous posteriors differ materially, the differing classification is genuine HMM sequence dependence. If neither the observation nor prior belief explains the difference, the fit deserves rejection.

In [19]:
oos_audit_events = audit_events.loc[audit_events["sample"].eq("oos")].copy()
slow_oos = oos_audit_events.loc[oos_audit_events["state_name"].eq("slow_R")].reset_index(drop=True)
fast_oos = oos_audit_events.loc[oos_audit_events["state_name"].eq("fast_R")].reset_index(drop=True)
if slow_oos.empty or fast_oos.empty:
    raise ValueError("Both HMM states must occur OOS for the nearest-pair audit.")

log_r_gaps = np.abs(
    fast_oos["log_r"].to_numpy()[:, None] - slow_oos["log_r"].to_numpy()[None, :]
)
nearest_slow_positions = log_r_gaps.argmin(axis=1)
pair_rows = []
for fast_position, slow_position in enumerate(nearest_slow_positions):
    fast_event = fast_oos.iloc[fast_position]
    slow_event = slow_oos.iloc[int(slow_position)]
    pair_rows.append({
        "log_r_gap": abs(float(fast_event["log_r"] - slow_event["log_r"])),
        "relative_r_gap": abs(float(fast_event["r"] - slow_event["r"]))
        / ((float(fast_event["r"]) + float(slow_event["r"])) / 2.0),
        "fast_event_id": int(fast_event["event_id"]),
        "slow_event_id": int(slow_event["event_id"]),
        "fast_time": fast_event["information_time"],
        "slow_time": slow_event["information_time"],
        "fast_r": fast_event["r"],
        "slow_r": slow_event["r"],
        "fast_t": int(fast_event["t"]),
        "slow_t": int(slow_event["t"]),
        "fast_tmv": fast_event["tmv"],
        "slow_tmv": slow_event["tmv"],
        "fast_previous_p": fast_event["previous_p_fast_r"],
        "slow_previous_p": slow_event["previous_p_fast_r"],
        "fast_current_p": fast_event["p_fast_r"],
        "slow_current_p": slow_event["p_fast_r"],
    })

nearest_opposite_state_pairs = pd.DataFrame(pair_rows).sort_values("log_r_gap").head(20)
nearest_opposite_state_pairs.style.format({
    "log_r_gap": "{:.6f}",
    "relative_r_gap": "{:.3%}",
    "fast_r": "{:.6e}",
    "slow_r": "{:.6e}",
    "fast_tmv": "{:.4f}",
    "slow_tmv": "{:.4f}",
    "fast_previous_p": "{:.4f}",
    "slow_previous_p": "{:.4f}",
    "fast_current_p": "{:.4f}",
    "slow_current_p": "{:.4f}",
})

,log_r_gap,relative_r_gap,fast_event_id,slow_event_id,fast_time,slow_time,fast_r,slow_r,fast_t,slow_t,fast_tmv,slow_tmv,fast_previous_p,slow_previous_p,fast_current_p,slow_current_p
66,0.000000,0.000%,5388,5500,2026-03-03 22:05:00+02:00,2026-03-18 14:20:00+02:00,9.767252e-05,9.767249e-05,12,25,1.0465,2.1802,0.9740,0.0017,0.5665,0.0273
184,0.000009,0.001%,6026,6195,2026-07-30 20:25:00+02:00,2026-10-01 13:15:00+02:00,2.449153e-04,2.449175e-04,12,10,2.6241,2.1868,0.2791,0.0062,0.5722,0.2325
58,0.000018,0.002%,5376,5989,2026-03-02 20:45:00+02:00,2026-07-16 19:35:00+02:00,2.359024e-04,2.359068e-04,17,5,3.5807,1.0532,0.7859,0.0028,0.8566,0.2109
102,0.000026,0.003%,5445,5467,2026-03-09 22:15:00+02:00,2026-03-11 16:55:00+02:00,1.561516e-04,1.561476e-04,23,10,3.2067,1.3942,0.9232,0.0251,0.7799,0.1056
79,0.000039,0.004%,5407,5936,2026-03-04 21:25:00+02:00,2026-06-30 15:35:00+02:00,1.429652e-04,1.429596e-04,8,13,1.0212,1.6594,0.9055,0.0314,0.7250,0.0905
142,0.000044,0.004%,5555,5278,2026-03-24 00:25:00+02:00,2026-02-09 03:45:00+02:00,1.691601e-04,1.691675e-04,29,7,4.3800,1.0573,0.8957,0.0005,0.7954,0.1032
30,0.000070,0.007%,5207,5322,2026-01-29 05:00:00+02:00,2026-02-19 19:00:00+02:00,8.743523e-05,8.742911e-05,51,62,3.9814,4.8398,0.9922,0.0001,0.5144,0.0202
147,0.000093,0.009%,5597,5844,2026-03-31 22:55:00+02:00,2026-06-08 08:55:00+02:00,2.618499e-04,2.618255e-04,13,5,3.0393,1.1689,0.5079,0.0093,0.7666,0.2680
85,0.000132,0.013%,5419,5652,2026-03-05 22:15:00+02:00,2026-04-10 20:20:00+02:00,1.903098e-04,1.902847e-04,14,6,2.3789,1.0194,0.6756,0.0094,0.7136,0.1442
156,0.000205,0.021%,5638,5879,2026-04-08 12:25:00+02:00,2026-06-15 04:55:00+02:00,3.111824e-04,3.112464e-04,7,14,1.9449,3.8906,0.8510,0.0003,0.9353,0.3354


# Audit 6: does the current state describe what happens next?

The previous audits explain the classification of event $n$. This audit moves the target forward by exactly one completed event. Every predictor below is available at `information_time_n`; every outcome belongs to event $n+1$ and is only used as a target.

The primary evidence is OOS. Train rows whose target crosses the train/OOS boundary are excluded. Pairs with non-increasing confirmation timestamps are also excluded because event $n+1$ would not be strictly later in physical time.

In [20]:
forward_events = audit_events.copy().reset_index(drop=True)
next_columns = {
    "event_id": "next_event_id",
    "information_time": "next_information_time",
    "sample": "next_sample",
    "state_name": "next_state_name",
    "r": "next_r",
    "log_r": "next_log_r",
    "t": "next_t",
    "tmv": "next_tmv",
    "direction": "next_direction",
}
for current_column, next_column in next_columns.items():
    forward_events[next_column] = forward_events[current_column].shift(-1)

information_positions = prices.index.searchsorted(
    pd.DatetimeIndex(forward_events["information_time"]), side="left"
)
forward_events["information_position"] = information_positions
forward_events["next_information_position"] = pd.Series(information_positions).shift(-1)
forward_events["next_confirmation_gap_bars"] = (
    forward_events["next_information_position"] - forward_events["information_position"]
)
forward_events["next_is_fast_r"] = forward_events["next_state_name"].eq("fast_R").astype(float)
forward_events.loc[forward_events["next_state_name"].isna(), "next_is_fast_r"] = np.nan

strictly_later = forward_events["next_information_time"] > forward_events["information_time"]
complete_target = forward_events[list(next_columns.values())].notna().all(axis=1)
forward_events["valid_forward_pair"] = strictly_later & complete_target
complete_forward_events = forward_events.loc[forward_events["valid_forward_pair"]].copy()

train_forward = complete_forward_events.loc[
    complete_forward_events["sample"].eq("train") & complete_forward_events["next_sample"].eq("train")
].copy()
oos_forward = complete_forward_events.loc[
    complete_forward_events["sample"].eq("oos") & complete_forward_events["next_sample"].eq("oos")
].copy()

assert (train_forward["next_information_time"] < oos_start_time).all()
assert (oos_forward["information_time"] >= oos_start_time).all()
assert (complete_forward_events["next_confirmation_gap_bars"] > 0).all()

excluded_non_later = int((complete_target & ~strictly_later).sum())
print(f"Valid train pairs: {len(train_forward):,}")
print(f"Valid OOS pairs: {len(oos_forward):,}")
print(f"Excluded non-later confirmation pairs: {excluded_non_later:,}")

Valid train pairs: 5,096
Valid OOS pairs: 1,274
Excluded non-later confirmation pairs: 0


In [21]:
oos_next_event_summary = oos_forward.groupby("state_name", sort=False).agg(
    current_event_count=("event_id", "size"),
    next_fast_rate=("next_is_fast_r", "mean"),
    next_r_mean=("next_r", "mean"),
    next_r_median=("next_r", "median"),
    next_r_p90=("next_r", lambda values: values.quantile(0.90)),
    next_t_mean=("next_t", "mean"),
    next_t_median=("next_t", "median"),
    next_tmv_mean=("next_tmv", "mean"),
    next_tmv_median=("next_tmv", "median"),
    next_confirmation_gap_bars_mean=("next_confirmation_gap_bars", "mean"),
    next_confirmation_gap_bars_median=("next_confirmation_gap_bars", "median"),
)
oos_state_transition_rates = pd.crosstab(
    oos_forward["state_name"],
    oos_forward["next_state_name"],
    normalize="index",
).rename_axis(index="current_state", columns="next_state")

display(oos_next_event_summary.style.format({
    "current_event_count": "{:,}",
    "next_fast_rate": "{:.2%}",
    "next_r_mean": "{:.6e}",
    "next_r_median": "{:.6e}",
    "next_r_p90": "{:.6e}",
    "next_t_mean": "{:.2f}",
    "next_t_median": "{:.1f}",
    "next_tmv_mean": "{:.4f}",
    "next_tmv_median": "{:.4f}",
    "next_confirmation_gap_bars_mean": "{:.2f}",
    "next_confirmation_gap_bars_median": "{:.1f}",
}))
display(oos_state_transition_rates.style.format("{:.2%}"))

,current_event_count,next_fast_rate,next_r_mean,next_r_median,next_r_p90,next_t_mean,next_t_median,next_tmv_mean,next_tmv_median,next_confirmation_gap_bars_mean,next_confirmation_gap_bars_median
state_name,,,,,,,,,,,
slow_R,"1,045",7.56%,1.193974e-04,6.917360e-05,2.427560e-04,54.75,34.0,2.3379,1.9077,54.21,33.0
fast_R,229,65.50%,2.386988e-04,1.638832e-04,5.490640e-04,25.26,12.0,2.2496,1.9446,27.74,15.0


next_state,fast_R,slow_R
current_state,,
fast_R,65.50%,34.50%
slow_R,7.56%,92.44%


In [22]:
next_event_outcomes_fig = plot_hmm_next_event_outcomes(complete_forward_events, sample="oos")
next_event_outcomes_fig.show()

## Does HMM history improve on current log(R)?

The next state is an internal HMM-defined target, so this is a coherence test—not evidence of trading value. Lower OOS Brier score and log loss are better; higher ROC AUC is better.

The comparison is `current_log_R` versus `current_log_R_plus_p_fast`. The direct HMM forecast propagates the current filtered probabilities through the fitted transition matrix before event $n+1$ is observed.

**Caution: this target is partly circular.** The filtered label at event $n+1$ is computed from $p(n)\,A$ and $R_{n+1}$. Adding `p_fast_r(n)` therefore helps predict it by construction, even when R has no persistence, so an improvement here does not show that the event history carries market information. Audit 6b below repeats the comparison on targets computed from the next event's R alone.

In [23]:
class NextStateForecast(StrEnum):
    TRAIN_PREVALENCE = "train_prevalence"
    CURRENT_LOG_R = "current_log_R"
    HMM_TRANSITION = "hmm_transition"
    CURRENT_LOG_R_PLUS_P_FAST = "current_log_R_plus_p_fast"


semantic_transition_matrix = hmm_model.transition_prob()[np.ix_(raw_state_order, raw_state_order)]
forward_events["hmm_transition_p_next_fast"] = semantic_probabilities @ semantic_transition_matrix[:, 1]
complete_forward_events["hmm_transition_p_next_fast"] = forward_events.loc[
    complete_forward_events.index, "hmm_transition_p_next_fast"
]
train_forward = complete_forward_events.loc[train_forward.index].copy()
oos_forward = complete_forward_events.loc[oos_forward.index].copy()

train_target = train_forward["next_is_fast_r"].astype(int).to_numpy()
oos_target = oos_forward["next_is_fast_r"].astype(int).to_numpy()
if np.unique(train_target).size != 2 or np.unique(oos_target).size != 2:
    raise ValueError("Both next-event states must occur in train and OOS for forecast comparison.")

log_r_forecaster = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1_000, random_state=HMM_RANDOM_STATE),
)
augmented_forecaster = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1_000, random_state=HMM_RANDOM_STATE),
)
log_r_forecaster.fit(train_forward[["log_r"]], train_target)
augmented_forecaster.fit(train_forward[["log_r", "p_fast_r"]], train_target)

forecast_probabilities = {
    NextStateForecast.TRAIN_PREVALENCE: np.full(len(oos_forward), train_target.mean()),
    NextStateForecast.CURRENT_LOG_R: log_r_forecaster.predict_proba(oos_forward[["log_r"]])[:, 1],
    NextStateForecast.HMM_TRANSITION: oos_forward["hmm_transition_p_next_fast"].to_numpy(),
    NextStateForecast.CURRENT_LOG_R_PLUS_P_FAST: augmented_forecaster.predict_proba(
        oos_forward[["log_r", "p_fast_r"]]
    )[:, 1],
}

forecast_rows = []
metric_rows = []
for forecast_name, probabilities in forecast_probabilities.items():
    clipped_probabilities = np.clip(probabilities, 1e-12, 1.0 - 1e-12)
    metric_rows.append({
        "model": forecast_name.value,
        "brier_score": brier_score_loss(oos_target, clipped_probabilities),
        "log_loss": log_loss(oos_target, clipped_probabilities, labels=[0, 1]),
        "roc_auc": roc_auc_score(oos_target, clipped_probabilities),
        "mean_predicted_fast_rate": clipped_probabilities.mean(),
        "observed_fast_rate": oos_target.mean(),
    })
    forecast_rows.append(pd.DataFrame({
        "event_id": oos_forward["event_id"].to_numpy(),
        "model": forecast_name.value,
        "predicted_probability": clipped_probabilities,
        "actual_next_fast": oos_target,
    }))

next_state_forecast_metrics = pd.DataFrame(metric_rows).sort_values("log_loss")
next_state_forecasts = pd.concat(forecast_rows, ignore_index=True)
next_state_forecast_metrics.style.format({
    "brier_score": "{:.6f}",
    "log_loss": "{:.6f}",
    "roc_auc": "{:.4f}",
    "mean_predicted_fast_rate": "{:.2%}",
    "observed_fast_rate": "{:.2%}",
})

,model,brier_score,log_loss,roc_auc,mean_predicted_fast_rate,observed_fast_rate
3,current_log_R_plus_p_fast,0.088913,0.296549,0.8935,19.99%,17.97%
2,hmm_transition,0.088965,0.306585,0.8968,22.63%,17.97%
1,current_log_R,0.102145,0.335138,0.8702,21.14%,17.97%
0,train_prevalence,0.160438,0.505474,0.5000,29.38%,17.97%


In [24]:
calibration_rows = []
for model_name, model_forecasts in next_state_forecasts.groupby("model", sort=False):
    model_forecasts = model_forecasts.copy()
    if model_forecasts["predicted_probability"].nunique() == 1:
        model_forecasts["probability_bin"] = "constant"
    else:
        model_forecasts["probability_bin"] = pd.qcut(
            model_forecasts["predicted_probability"], q=10, duplicates="drop"
        ).astype(str)
    model_calibration = model_forecasts.groupby("probability_bin", observed=True).agg(
        mean_predicted_probability=("predicted_probability", "mean"),
        observed_fast_rate=("actual_next_fast", "mean"),
        event_count=("event_id", "size"),
    ).reset_index()
    model_calibration.insert(0, "model", model_name)
    calibration_rows.append(model_calibration)

next_state_calibration = pd.concat(calibration_rows, ignore_index=True)
next_state_calibration_fig = plot_hmm_next_state_calibration(next_state_calibration)
next_state_calibration_fig.show()

# Audit 6b: model-free next-event targets

Audit 6 predicts the next **filtered label**. That target comes from the same HMM: the label at event $n+1$ is computed from $p(n)\,A$ and $R_{n+1}$. Adding `p_fast_r(n)` as a predictor therefore helps predict it by construction, even if R had no persistence at all. Audit 6 alone cannot show that HMM history carries information about the market.

This audit removes the circularity. Both targets are computed from the next event's observed R only:

1. **`next_log_R`** (regression): the next completed event's log R.
2. **`next_R_above_train_median`** (classification): whether the next event's R exceeds the median R of the training events.

Every forecast uses only information available at `information_time_n`:

- the training mean (regression) or training prevalence (classification);
- current log R;
- current and previous log R (one lag: the direct, model-free use of event history);
- current log R plus `p_fast_r`, and one lag plus `p_fast_r`;
- the HMM's own one-step predictive: filtered probabilities propagated through the transition matrix, then through each state's fitted log-R emission.

Two comparisons decide what history adds. One lag of log R is history used directly, with no model; the HMM state is credited only if `p_fast_r` adds beyond that lag, or if `hmm_one_step_predictive` beats current log R. The last table puts a moving-block bootstrap interval (blocks of 50 events) on each loss difference, because neighbouring events are dependent; a difference counts only when its interval excludes zero. Lower MSE, MAE, Brier score and log loss are better; higher skill and ROC AUC are better.

In [25]:
class ModelFreeForecast(StrEnum):
    TRAIN_BASELINE = "train_mean_or_prevalence"
    CURRENT_LOG_R = "current_log_R"
    HMM_ONE_STEP = "hmm_one_step_predictive"
    CURRENT_LOG_R_PLUS_P_FAST = "current_log_R_plus_p_fast"
    CURRENT_AND_PREVIOUS_LOG_R = "current_and_previous_log_R"
    CURRENT_AND_PREVIOUS_PLUS_P_FAST = "current_and_previous_log_R_plus_p_fast"


raw_transition_matrix = hmm_model.transition_prob()
emission_scales = np.sqrt(np.array([float(np.ravel(parameters["covs"])[0]) for parameters in hmm_model.parameters]))
next_state_probabilities = filtered_probabilities @ raw_transition_matrix  # p(S_n+1 | R_1..R_n), raw state order
training_median_log_r = float(hmm_events["log_r"].iloc[:train_event_count].median())

forward_events["hmm_expected_next_log_r"] = next_state_probabilities @ emission_locations
forward_events["hmm_p_next_above_median"] = next_state_probabilities @ norm.sf(
    training_median_log_r, loc=emission_locations, scale=emission_scales
)
forward_events["next_above_train_median"] = (forward_events["next_log_r"] > training_median_log_r).astype(float)
forward_events.loc[forward_events["next_log_r"].isna(), "next_above_train_median"] = np.nan

# The previous event's log R completed before the current DCC, so it is known at information_time_n.
forward_events["previous_log_r"] = forward_events["log_r"].shift(1)
model_free_train = forward_events.loc[train_forward.index]
model_free_train = model_free_train.loc[model_free_train["previous_log_r"].notna()]  # drops the first event only
model_free_oos = forward_events.loc[oos_forward.index]
if model_free_oos["previous_log_r"].isna().any():
    raise AssertionError("Every out-of-sample event must have a previous completed event.")
predictor_sets = {
    ModelFreeForecast.CURRENT_LOG_R: ["log_r"],
    ModelFreeForecast.CURRENT_LOG_R_PLUS_P_FAST: ["log_r", "p_fast_r"],
    ModelFreeForecast.CURRENT_AND_PREVIOUS_LOG_R: ["log_r", "previous_log_r"],
    ModelFreeForecast.CURRENT_AND_PREVIOUS_PLUS_P_FAST: ["log_r", "previous_log_r", "p_fast_r"],
}

# Target 1: next log R (regression).
y_train_log_r = model_free_train["next_log_r"].to_numpy(dtype=float)
y_oos_log_r = model_free_oos["next_log_r"].to_numpy(dtype=float)
regression_predictions = {
    ModelFreeForecast.TRAIN_BASELINE: np.full(len(model_free_oos), y_train_log_r.mean()),
    ModelFreeForecast.HMM_ONE_STEP: model_free_oos["hmm_expected_next_log_r"].to_numpy(dtype=float),
}
for forecast, columns in predictor_sets.items():
    regressor = LinearRegression().fit(model_free_train[columns], y_train_log_r)
    regression_predictions[forecast] = regressor.predict(model_free_oos[columns])

baseline_mse = mean_squared_error(y_oos_log_r, regression_predictions[ModelFreeForecast.TRAIN_BASELINE])
next_log_r_metrics = pd.DataFrame([
    {
        "model": forecast.value,
        "oos_mse": mean_squared_error(y_oos_log_r, predictions),
        "oos_mae": mean_absolute_error(y_oos_log_r, predictions),
        "skill_vs_train_mean": 1.0 - mean_squared_error(y_oos_log_r, predictions) / baseline_mse,
    }
    for forecast, predictions in regression_predictions.items()
]).sort_values("oos_mse")

# Target 2: is the next R above the training median? (classification)
y_train_above = model_free_train["next_above_train_median"].astype(int).to_numpy()
y_oos_above = model_free_oos["next_above_train_median"].astype(int).to_numpy()
if np.unique(y_train_above).size != 2 or np.unique(y_oos_above).size != 2:
    raise ValueError("Both outcomes of the median-split target must occur in train and OOS.")

classification_probabilities = {
    ModelFreeForecast.TRAIN_BASELINE: np.full(len(model_free_oos), y_train_above.mean()),
    ModelFreeForecast.HMM_ONE_STEP: model_free_oos["hmm_p_next_above_median"].to_numpy(dtype=float),
}
for forecast, columns in predictor_sets.items():
    classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1_000, random_state=HMM_RANDOM_STATE),
    )
    classifier.fit(model_free_train[columns], y_train_above)
    classification_probabilities[forecast] = classifier.predict_proba(model_free_oos[columns])[:, 1]

above_median_rows = []
for forecast, probabilities in classification_probabilities.items():
    clipped_probabilities = np.clip(probabilities, 1e-12, 1.0 - 1e-12)
    above_median_rows.append({
        "model": forecast.value,
        "brier_score": brier_score_loss(y_oos_above, clipped_probabilities),
        "log_loss": log_loss(y_oos_above, clipped_probabilities, labels=[0, 1]),
        "roc_auc": roc_auc_score(y_oos_above, clipped_probabilities),
        "mean_predicted_rate": clipped_probabilities.mean(),
        "observed_rate": y_oos_above.mean(),
    })
next_above_median_metrics = pd.DataFrame(above_median_rows).sort_values("log_loss")

print(f"OOS pairs: {len(model_free_oos):,} | training median log(R): {training_median_log_r:.4f}")
display(next_log_r_metrics.style.format({
    "oos_mse": "{:.6f}",
    "oos_mae": "{:.6f}",
    "skill_vs_train_mean": "{:.2%}",
}))
display(next_above_median_metrics.style.format({
    "brier_score": "{:.6f}",
    "log_loss": "{:.6f}",
    "roc_auc": "{:.4f}",
    "mean_predicted_rate": "{:.2%}",
    "observed_rate": "{:.2%}",
}))

# Does history add anything, and does the HMM add beyond plain history?
# Loss differences are serially dependent, so intervals come from a moving-block bootstrap.
GAIN_BLOCK_EVENTS = 50
GAIN_BOOTSTRAP_DRAWS = 2_000


def block_bootstrap_mean_gain(baseline_loss: np.ndarray, candidate_loss: np.ndarray) -> tuple[float, float, float]:
    """Mean of (baseline loss - candidate loss) and its 95% moving-block bootstrap interval."""
    differences = np.asarray(baseline_loss, dtype=float) - np.asarray(candidate_loss, dtype=float)
    rows = len(differences)
    rng = np.random.default_rng(HMM_RANDOM_STATE)
    block_count = int(np.ceil(rows / GAIN_BLOCK_EVENTS))
    offsets = np.arange(GAIN_BLOCK_EVENTS)
    draws = np.empty(GAIN_BOOTSTRAP_DRAWS)
    for draw in range(GAIN_BOOTSTRAP_DRAWS):
        starts = rng.integers(0, rows - GAIN_BLOCK_EVENTS + 1, size=block_count)
        draws[draw] = differences[(starts[:, None] + offsets[None, :]).ravel()[:rows]].mean()
    low, high = np.quantile(draws, [0.025, 0.975])
    return float(differences.mean()), float(low), float(high)


GAIN_COMPARISONS = [
    ("one lag of log R beyond current log R", ModelFreeForecast.CURRENT_LOG_R, ModelFreeForecast.CURRENT_AND_PREVIOUS_LOG_R),
    ("p_fast_r beyond current log R", ModelFreeForecast.CURRENT_LOG_R, ModelFreeForecast.CURRENT_LOG_R_PLUS_P_FAST),
    ("p_fast_r beyond one lag", ModelFreeForecast.CURRENT_AND_PREVIOUS_LOG_R, ModelFreeForecast.CURRENT_AND_PREVIOUS_PLUS_P_FAST),
    ("HMM one-step forecast vs current log R", ModelFreeForecast.CURRENT_LOG_R, ModelFreeForecast.HMM_ONE_STEP),
]
gain_rows = []
for target, losses in (
    ("next_log_R squared error", {
        forecast: (y_oos_log_r - predictions) ** 2 for forecast, predictions in regression_predictions.items()
    }),
    ("next_R_above_median Brier", {
        forecast: (y_oos_above - np.clip(probabilities, 1e-12, 1.0 - 1e-12)) ** 2
        for forecast, probabilities in classification_probabilities.items()
    }),
):
    for label, baseline, candidate in GAIN_COMPARISONS:
        gain, low, high = block_bootstrap_mean_gain(losses[baseline], losses[candidate])
        gain_rows.append({
            "target": target,
            "comparison": label,
            "mean_loss_reduction": gain,
            "ci95_low": low,
            "ci95_high": high,
            "verdict": "better" if low > 0 else ("worse" if high < 0 else "no detectable difference"),
        })
history_gains = pd.DataFrame(gain_rows).set_index(["target", "comparison"])
print(f"Positive = lower loss than the baseline; {GAIN_BLOCK_EVENTS}-event blocks, {GAIN_BOOTSTRAP_DRAWS:,} draws.")
display(history_gains.style.format({
    "mean_loss_reduction": "{:+.5f}",
    "ci95_low": "{:+.5f}",
    "ci95_high": "{:+.5f}",
}))

OOS pairs: 1,274 | training median log(R): -9.1207


,model,oos_mse,oos_mae,skill_vs_train_mean
5,current_and_previous_log_R_plus_p_fast,0.846022,0.715969,23.32%
4,current_and_previous_log_R,0.847224,0.715368,23.21%
3,current_log_R_plus_p_fast,0.848827,0.717285,23.06%
2,current_log_R,0.865359,0.722476,21.56%
1,hmm_one_step_predictive,0.897420,0.750755,18.66%
0,train_mean_or_prevalence,1.103279,0.849723,0.00%


,model,brier_score,log_loss,roc_auc,mean_predicted_rate,observed_rate
4,current_and_previous_log_R,0.196114,0.575630,0.7492,43.14%,37.13%
5,current_and_previous_log_R_plus_p_fast,0.196293,0.578327,0.7512,43.16%,37.13%
3,current_log_R_plus_p_fast,0.196648,0.579431,0.7510,43.23%,37.13%
2,current_log_R,0.198253,0.581109,0.7461,43.66%,37.13%
1,hmm_one_step_predictive,0.207435,0.605577,0.7477,43.68%,37.13%
0,train_mean_or_prevalence,0.249975,0.693097,0.5000,49.99%,37.13%


Positive = lower loss than the baseline; 50-event blocks, 2,000 draws.


## How to interpret Audits 6 and 6b

1. First compare the OOS next-event table and box plots. If the next-event distributions are almost identical after `slow_R` and `fast_R`, the current state is descriptive rather than forward-looking.
2. Compare `hmm_transition` with `train_prevalence` in Audit 6. Improvement shows that the fitted transition matrix is persistent; on its own next label, that is a coherence property of the model.
3. In Audit 6, `current_log_R_plus_p_fast` beating `current_log_R` is expected by construction, because the target label is computed from $p(n)$. It is **not** evidence that history carries market information.
4. Use Audit 6b for that test, and read its last table. Credit the event history when one lag of log R beats current log R with an interval above zero. Credit the HMM state only when `p_fast_r` beats the one-lag model the same way, or when `hmm_one_step_predictive` beats current log R. Otherwise the state describes the completed event; it does not add a forecast beyond plain history.
5. Inspect calibration, not only discrimination. A high ROC AUC with poor probability calibration is not a trustworthy probability forecast.
6. This remains an event-structure audit. It does not establish return predictability or trading utility.

# Audit 7: recent-event economic casebook

This casebook completes the qualitative step in the learning plan. It selects one recent, visually manageable OOS example of each filtered-state transition.

At the current DCC we know the confirmed extreme, completed-leg direction, TMV, T, R, filtered velocity state, and the new live DC mode. The following extreme, following DCC, following state, and post-confirmation excursion are shown only in hindsight. They are outcomes to inspect—not information the current decision could use.

In [26]:
class CasebookTransition(StrEnum):
    SLOW_TO_SLOW = "slow_R → slow_R"
    SLOW_TO_FAST = "slow_R → fast_R"
    FAST_TO_FAST = "fast_R → fast_R"
    FAST_TO_SLOW = "fast_R → slow_R"


CASEBOOK_MAX_LIFECYCLE_BARS = 800
CASEBOOK_PADDING_BARS = 12
TRANSITION_STATES = {
    CasebookTransition.SLOW_TO_SLOW: ("slow_R", "slow_R"),
    CasebookTransition.SLOW_TO_FAST: ("slow_R", "fast_R"),
    CasebookTransition.FAST_TO_FAST: ("fast_R", "fast_R"),
    CasebookTransition.FAST_TO_SLOW: ("fast_R", "slow_R"),
}

casebook_candidates = audit_events.copy().reset_index(drop=True)
casebook_candidates["previous_state_name"] = casebook_candidates["state_name"].shift(1)
casebook_candidates["previous_event_id"] = casebook_candidates["event_id"].shift(1)
casebook_next_columns = [
    "event_id", "state_name", "ext_end_idx", "ext_end_price", "ext_end_pos",
    "dcc_idx", "dcc_price", "dcc_pos", "tmv", "t", "r",
]
for column in casebook_next_columns:
    casebook_candidates[f"next_{column}"] = casebook_candidates[column].shift(-1)

# The DCC's direction is the opposite of the completed leg; the ledger states it as new_live_mode.
casebook_candidates["live_mode"] = casebook_candidates["new_live_mode"]
casebook_candidates["confirmation_type"] = casebook_candidates["live_mode"] + "ward DCC"
casebook_candidates["lifecycle_bar_count"] = (
    casebook_candidates["next_dcc_pos"] - casebook_candidates["ext_start_pos"] + 1
)
casebook_candidates["post_confirmation_bars"] = (
    casebook_candidates["next_ext_end_pos"] - casebook_candidates["dcc_pos"]
)
casebook_candidates["post_confirmation_excursion_theta"] = np.where(
    casebook_candidates["live_mode"].eq("up"),
    (casebook_candidates["next_ext_end_price"] / casebook_candidates["dcc_price"] - 1.0) / THETA,
    (1.0 - casebook_candidates["next_ext_end_price"] / casebook_candidates["dcc_price"]) / THETA,
)

eligible_casebook_events = casebook_candidates.loc[
    casebook_candidates["sample"].eq("oos")
    & casebook_candidates["previous_state_name"].notna()
    & casebook_candidates["next_dcc_idx"].notna()
    & casebook_candidates["next_dcc_idx"].gt(casebook_candidates["dcc_idx"])
].copy()

selected_casebook_rows = []
missing_transitions = []
for transition, (previous_state, current_state) in TRANSITION_STATES.items():
    transition_candidates = eligible_casebook_events.loc[
        eligible_casebook_events["previous_state_name"].eq(previous_state)
        & eligible_casebook_events["state_name"].eq(current_state)
    ]
    if transition_candidates.empty:
        missing_transitions.append(transition.value)
        continue
    manageable_candidates = transition_candidates.loc[
        transition_candidates["lifecycle_bar_count"].le(CASEBOOK_MAX_LIFECYCLE_BARS)
    ]
    selection_pool = manageable_candidates if not manageable_candidates.empty else transition_candidates
    selected_event = selection_pool.iloc[-1].copy()
    selected_event["transition_label"] = transition.value
    selected_casebook_rows.append(selected_event)

if missing_transitions:
    raise ValueError(f"No eligible recent OOS examples for transitions: {missing_transitions}")

casebook_events = pd.DataFrame(selected_casebook_rows).reset_index(drop=True)
casebook_events["window_start_pos"] = (
    casebook_events["ext_start_pos"] - CASEBOOK_PADDING_BARS
).clip(lower=0).astype(int)
casebook_events["window_end_pos"] = (
    casebook_events["next_dcc_pos"] + CASEBOOK_PADDING_BARS
).clip(upper=len(prices) - 1).astype(int)
casebook_events["window_start"] = prices.index[casebook_events["window_start_pos"]]
casebook_events["window_end"] = prices.index[casebook_events["window_end_pos"]]

assert (casebook_events["dcc_idx"] == casebook_events["information_time"]).all()
assert (casebook_events["next_dcc_idx"] > casebook_events["dcc_idx"]).all()
casebook_events[["transition_label", "event_id", "information_time", "lifecycle_bar_count"]]

,transition_label,event_id,information_time,lifecycle_bar_count
0,slow_R → slow_R,6371,2026-11-06 17:55:00+02:00,119.0
1,slow_R → fast_R,6364,2026-11-05 18:45:00+02:00,36.0
2,fast_R → fast_R,6353,2026-11-04 06:50:00+02:00,52.0
3,fast_R → slow_R,6365,2026-11-05 20:40:00+02:00,89.0


In [27]:
casebook_ledger_columns = [
    "transition_label", "event_id", "information_time", "direction", "confirmation_type",
    "live_mode", "state_name", "tmv", "t", "r", "p_fast_r", "state_probability",
    "post_confirmation_excursion_theta", "post_confirmation_bars", "next_event_id",
    "next_state_name", "next_tmv", "next_t", "next_r", "next_dcc_idx",
]
casebook_ledger = casebook_events[casebook_ledger_columns].copy()
casebook_ledger.style.format({
    "event_id": "{:.0f}",
    "tmv": "{:.4f}",
    "t": "{:.0f}",
    "r": "{:.6e}",
    "p_fast_r": "{:.3f}",
    "state_probability": "{:.3f}",
    "post_confirmation_excursion_theta": "{:.2f}θ",
    "post_confirmation_bars": "{:.0f}",
    "next_event_id": "{:.0f}",
    "next_tmv": "{:.4f}",
    "next_t": "{:.0f}",
    "next_r": "{:.6e}",
})

,transition_label,event_id,information_time,direction,confirmation_type,live_mode,state_name,tmv,t,r,p_fast_r,state_probability,post_confirmation_excursion_theta,post_confirmation_bars,next_event_id,next_state_name,next_tmv,next_t,next_r,next_dcc_idx
0,slow_R → slow_R,6371,2026-11-06 17:55:00+02:00,up,downward DCC,down,slow_R,2.1254,12,1.983699e-04,0.173,0.827,0.66θ,72,6372,slow_R,1.7763,79,2.518283e-05,2026-11-09 02:10:00+02:00
1,slow_R → fast_R,6364,2026-11-05 18:45:00+02:00,down,upward DCC,up,fast_R,1.3276,4,3.717224e-04,0.638,0.638,0.64θ,13,6365,slow_R,1.9983,21,1.065769e-04,2026-11-05 20:40:00+02:00
2,fast_R → fast_R,6353,2026-11-04 06:50:00+02:00,up,downward DCC,down,fast_R,1.5018,14,1.201410e-04,0.586,0.586,1.23θ,28,6354,slow_R,2.5305,32,8.856624e-05,2026-11-04 09:35:00+02:00
3,fast_R → slow_R,6365,2026-11-05 20:40:00+02:00,up,downward DCC,down,slow_R,1.9983,21,1.065769e-04,0.335,0.665,0.83θ,29,6366,slow_R,1.9541,39,5.611876e-05,2026-11-06 01:25:00+02:00


In [28]:
recent_casebook_fig = plot_hmm_recent_event_casebook(prices, casebook_events)
recent_casebook_fig.show()

## Reading each case without look-ahead

At the colored triangular DCC marker, say the following out loud:

1. **What ended?** The green or red extreme-to-extreme segment is the completed leg. Its direction is retrospective.
2. **What is the mode now?** The opposite DCC has switched the live DC mode. A completed up leg produces a downward DCC and live down mode; a completed down leg produces an upward DCC and live up mode.
3. **How quickly did the completed leg develop?** TMV, T, and R determine whether its normalized velocity state is `slow_R` or `fast_R`.
4. **What does the shading mean?** It carries that completed-event velocity state forward until the next DCC. It is not a fresh classification on every bar.
5. **What was unknown?** The purple next extreme and black next DCC were unknowable at the current DCC. Use them only to judge whether the state was followed by economically different behaviour.

A precise market description at the current marker is: *The prior [up/down] leg completed with [slow/fast] normalized velocity; the opposite DCC has now put the process into [down/up] mode.*